# SAPLMA

Fit a three-hidden-layer classifier to token activations near the end of the decoder. The experiment adapts the method to token-level code defect detection. [Paper](https://arxiv.org/abs/2304.13734)

Train an independent probe on each benchmark. Line and sample scores use the maximum annotated token score. F1 thresholds come from validation.

In [1]:
from pathlib import Path
import sys

if Path("../src/traces.py").is_file():
    sys.path.insert(0, str(Path.cwd().parent))

In [2]:
import pandas as pd
import torch
from IPython.display import display

from datasets.benchmarks import DATASETS, PILOT_LIMITS
from src.evaluation import TraceStore, evaluate
from src.methods import SAPLMA

pilot = False
limits = PILOT_LIMITS if pilot else None
torch.set_num_threads(min(4, torch.get_num_threads()))
store = TraceStore(quantize=True)

In [3]:
results = []
for benchmark in DATASETS:
    splits = store.load_splits(benchmark, limits)
    results.extend(
        {"benchmark": benchmark, **row}
        for row in evaluate(SAPLMA(), splits)
    )

metrics = pd.DataFrame(results).set_index(["benchmark", "level"])
display(metrics.round(4))

n  positives  positive_rate  pr_auc  \
benchmark      level                                            
mbpp           token   3373        660         0.1957  0.6876   
               line     509         54         0.1061  0.7372   
               sample    74         31         0.4189  0.8115   
humaneval      token   3714        665         0.1791  0.4879   
               line     484         55         0.1136  0.5976   
               sample    46         19         0.4130  0.6066   
humaneval_java token   1157        205         0.1772  0.2952   
               line     155         15         0.0968  0.3469   
               sample    36         11         0.3056  0.3377   
defects4j      token   3901       2220         0.5691  0.9228   
               line     448        148         0.3304  0.8633   
               sample    86         35         0.4070  0.6343   

                       average_precision  roc_auc  precision  recall      f1  \
benchmark      level                                                           
mbpp           token              0.6881   0.9132     0.5550  0.8106  0.6589   
               line               0.7407   0.9652     0.6667  0.7407  0.7018   
               sample             0.8163   0.8815     0.7073  0.9355  0.8056   
humaneval      token              0.4884   0.8154     0.1994  1.0000  0.3325   
               line               0.6008   0.9094     0.1682  1.0000  0.2880   
               sample             0.6167   0.6277     0.4516  0.7368  0.5600   
humaneval_java token              0.2987   0.6994     0.2881  0.4146  0.3400   
               line               0.3753   0.8929     0.3243  0.8000  0.4615   
               sample             0.3729   0.6000     0.3913  0.8182  0.5294   
defects4j      token              0.9229   0.8968     0.7227  0.9428  0.8182   
               line               0.8638   0.9313     0.6295  0.9527  0.7581   
               sample             0.6398   0.6717     0.4444  0.8000  0.5714   

                       threshold  
benchmark      level              
mbpp           token      0.0176  
               line       0.9489  
               sample     0.9489  
humaneval      token      0.0000  
               line       0.0005  
               sample     0.6975  
humaneval_java token      0.3360  
               line       0.6450  
               sample     0.7162  
defects4j      token      0.0954  
               line       0.5567  
               sample     0.7505